**Navigation** : [carnet source Python](Constructeur-01-Lois-Comme-Taches-Python.ipynb) | [Index de la série](README.md)

# Théorie du constructeur — les lois comme tâches possibles ou impossibles (C#)

**Série** : Physique / Constructeur · **Noyau** : .NET (C#)

Jumeau C# de [Constructeur-01 — les lois comme tâches possibles ou impossibles (Python)](Constructeur-01-Lois-Comme-Taches-Python.ipynb) : même fil pédagogique, mêmes entrées, mêmes verdicts — rejoués ici en **.NET natif**, BCL seule, aucun paquet externe ni NuGet.

La physique fondamentale s'énonce d'habitude en **dynamique** : des lois de mouvement qui
décrivent comment un état évolue, une fois les conditions initiales données. La
**théorie du constructeur** (Deutsch, 2013 ; Deutsch & Marletto, 2015) opère une inversion :
les lois fondamentales sont des énoncés sur **quelles transformations physiques sont
possibles, lesquelles sont impossibles, et pourquoi**.

> « Les lois sont sur lesquelles transformations physiques sont possibles et lesquelles
> sont impossibles, et pourquoi. » — constructortheory.org

Ce carnet distille ce changement de point de vue. Il est volontairement **conceptuel** :
les cellules de code illustrent les notions (tâche, constructeur, possible/impossible,
support d'information) sur des systèmes jouets, en C# sans dépendance.

## Le vocabulaire : tâche, substrat, état, attribut, constructeur

Le site d'origine illustre avec un **dé** :

- **substrat** — l'objet qui est transformé (le dé) ;
- **état** — les six faces possibles du dé ;
- **attribut** — un ensemble d'états (« la face est 3 », « la face est paire ») ;
- **tâche** — la spécification d'une transformation du substrat, d'un attribut d'entrée
  vers un attribut de sortie (ici : 3 → 4) ;
- **constructeur** — « un objet qui peut accomplir la tâche **et qui reste le même**
  après l'avoir accomplie » : il conserve sa capacité à la produire, donc il la répète ;
- **possible / impossible** — une tâche est **impossible** si « aucun constructeur ne
  peut l'accomplir à une précision arbitraire » ; sinon elle est **possible**.

Subtilité importante (FAQ du site) : « les lois de la théorie du constructeur ne sont
**pas** des lois sur les constructeurs ». Le constructeur est l'objet qui *témoigne* de la
possibilité d'une tâche ; la loi, elle, porte sur la **classe entière** des constructeurs
possibles. C'est pourquoi l'impossibilité ne se démontre jamais en constatant qu'un
constructeur particulier échoue.

En C#, ces deux notions se portent sans effort : une tâche est un triplet
(nom, entrée, sortie), un constructeur est une **règle** état → état, et « accomplir »
est une comparaison sur l'état de sortie.

In [1]:
// Illustration 1 -- une tache sur un substrat (le de)
// Le substrat a 6 etats ; une tache specifie une transformation d'attribut.
using System;
using System.Collections.Generic;
using System.Numerics;

// Une tache : specifier une transformation d'un etat d'entree vers un etat de sortie.
sealed class Tache
{
    public string Nom { get; }
    public string Entree { get; }
    public string Sortie { get; }

    public Tache(string nom, string entree, string sortie)
    {
        Nom = nom; Entree = entree; Sortie = sortie;
    }

    // True si le constructeur amene l'etat d'entree vers l'etat de sortie.
    public bool AccompliePar(Constructeur constructeur, string etat)
        => constructeur.Transformer(etat) == Sortie;
}

// Un constructeur : une regle etat -> etat. Inchange apres la tache, donc repetable.
sealed class Constructeur
{
    public string Nom { get; }
    public IReadOnlyDictionary<string, string> Regle { get; }

    public Constructeur(string nom, IReadOnlyDictionary<string, string> regle)
    {
        Nom = nom; Regle = regle;
    }

    public string Transformer(string etat)
        => Regle.TryGetValue(etat, out var sortie) ? sortie : etat;
}

// --- Exactitude et verification des valeurs rejouees ------------------------
// Rational : rationnel exact reduit, numerateur et denominateur BigInteger.
// Il porte les valeurs attendues exactement representables (1/2, 0/1, -1/2).
sealed class Rational
{
    public BigInteger Numerateur { get; }
    public BigInteger Denominateur { get; }

    public Rational(long numerateur, long denominateur)
    {
        if (denominateur == 0) throw new ArgumentException("denominateur nul");
        var n = new BigInteger(numerateur);
        var d = new BigInteger(denominateur);
        if (d < 0) { n = -n; d = -d; }
        var g = BigInteger.GreatestCommonDivisor(BigInteger.Abs(n), d);
        Numerateur = n / g;
        Denominateur = d / g;
    }

    // Image double exacte du rationnel (les cibles utilisees ici sont
    // exactement representables) ; la comparaison se fait par tolerance.
    public double ToDouble() => (double)Numerateur / (double)Denominateur;

    public override string ToString()
        => Denominateur == 1 ? Numerateur.ToString() : Numerateur + "/" + Denominateur;
}

// Verification : chaque valeur rejouee est verifiee par une assertion qui
// LEVE si elle differe de l'attendu. Un booleen imprime ne suffit pas.
static class Verification
{
    public static void Verifie(string nom, bool obtenu, bool attendu)
    {
        if (obtenu != attendu)
            throw new InvalidOperationException($"{nom} : obtenu {obtenu}, attendu {attendu}");
    }

    public static void Verifie(string nom, int obtenu, int attendu)
    {
        if (obtenu != attendu)
            throw new InvalidOperationException($"{nom} : obtenu {obtenu}, attendu {attendu}");
    }

    public static void Verifie(string nom, string obtenu, string attendu)
    {
        if (obtenu != attendu)
            throw new InvalidOperationException($"{nom} : obtenu \"{obtenu}\", attendu \"{attendu}\"");
    }

    // Cible exacte : rationnel reduit. Aucune egalite flottante -- la
    // comparaison passe par l'image double exacte et une tolerance explicite.
    public static void Verifie(string nom, double obtenu, Rational attendu, double tolerance)
    {
        double cible = attendu.ToDouble();
        if (Math.Abs(obtenu - cible) > tolerance)
            throw new InvalidOperationException(
                $"{nom} : obtenu {obtenu:R}, attendu {attendu} ({cible:R}) a {tolerance:R} pres");
    }

    // Crite irrationnelle (racine d'un polynome) : la tolerance porte
    // l'incertitude du calcul, pas une egalite flottante.
    public static void Verifie(string nom, double obtenu, double attendu, double tolerance)
    {
        if (Math.Abs(obtenu - attendu) > tolerance)
            throw new InvalidOperationException(
                $"{nom} : obtenu {obtenu:R}, attendu {attendu:R} a {tolerance:R} pres");
    }
}

var etats = new[] { "1", "2", "3", "4", "5", "6" };
var tache = new Tache("3 -> 4", "3", "4");
var c = new Constructeur("c_3vers4", new Dictionary<string, string> { ["3"] = "4" });

Console.WriteLine("Tache : " + tache.Nom);
Console.WriteLine("Accomplie par " + c.Nom + " : " + tache.AccompliePar(c, tache.Entree));
Console.WriteLine("Un constructeur qui ne fait rien l'accomplit-il ? " +
    tache.AccompliePar(new Constructeur("rien", new Dictionary<string, string>()), tache.Entree));
Console.WriteLine("Substrat : " + etats.Length + " etats possibles");

// --- Verification des valeurs rejouees -------------------------------------
Verification.Verifie("tache : entree 3", tache.Entree, "3");
Verification.Verifie("tache : sortie 4", tache.Sortie, "4");
Verification.Verifie("c_3vers4 accomplit la tache 3 -> 4", tache.AccompliePar(c, tache.Entree), true);
Verification.Verifie("un constructeur vide n'accomplit pas la tache",
    tache.AccompliePar(new Constructeur("rien", new Dictionary<string, string>()), tache.Entree), false);
Verification.Verifie("c_3vers4 transforme l'etat 3 en 4", c.Transformer("3"), "4");
Verification.Verifie("un etat absent de la regle reste inchange", c.Transformer("5"), "5");
Verification.Verifie("le substrat compte six etats", etats.Length, 6);

// Temoin negatif : le verificateur DOIT lever sur une valeur fausse. Sans ce
// temoin, un verificateur devenu muet passerait toutes les assertions.
bool leveSurValeurFausse = false;
try
{
    Verification.Verifie("temoin negatif", 0.5, new Rational(0, 1), 1e-12);
}
catch (InvalidOperationException)
{
    leveSurValeurFausse = true;
}
if (!leveSurValeurFausse)
    throw new InvalidOperationException("Le verificateur ne leve pas sur une valeur fausse.");


Tache : 3 -> 4


Accomplie par c_3vers4 : True


Un constructeur qui ne fait rien l'accomplit-il ? False


Substrat : 6 etats possibles


## L'inversion : de la dynamique aux principes

La physique traditionnelle a pour objets « les trajectoires, les lois dynamiques et les
conditions initiales » : elle dit ce qui arrivera plus tard, sachant l'état actuel. La
théorie du constructeur renverse l'ordre — « les lois dynamiques et les trajectoires
arrivent bien plus tard » :

- les lois sont des **principes** : « quelles tâches sont possibles, lesquelles sont
  impossibles, et pourquoi » ;
- ils **sous-tendent et complètent** les théories fondamentales existantes, sans les
  remplacer ;
- la description traditionnelle (conditions initiales + lois de mouvement) **peut être
  retrouvée** dans de nombreux cas ;
- le gain : énoncer **exactement**, sans approximation, des domaines « considérés comme
  intrinsèquement approximatifs » — l'information, la vie, la probabilité, la
  thermodynamique.

Le motif : des entités comme l'information ou la vie sont habituellement décrites par des
approximations (thermodynamique statistique, sélection naturelle), parce qu'on les croit
irréductiblement complexes. Le langage constructeur les décrit par des **énoncés
d'impossibilité exacts**, sans passer par une dynamique.

## La conservation comme impossibilité

Dans ce langage, « la loi de conservation de l'énergie énonce que **certains
constructeurs sont impossibles** » — en particulier le perpétuum mobile de première
espèce : un constructeur qui rend du travail et revient à son état initial **sans**
apport d'énergie. La conséquence en découle « sans jamais référer à une machine à
mouvement perpétuel précise ».

In [2]:
// Illustration 2 -- conservation de l'energie = tache impossible
// Un perpetuum mobile de 1re espece est un constructeur qui rend du travail
// et revient a son etat initial SANS apport d'energie : impossible.

// True si le constructeur fabrique de l'energie de rien (interdit).
static class Conservation
{
    public static bool EstPerpetuumMobile1reEspece(double apportEnergie, double travailRendu)
        => apportEnergie < travailRendu;
}

Console.WriteLine("Un moteur qui rend 10 J sans en recevoir est-il un perpetuum mobile ? " +
    Conservation.EstPerpetuumMobile1reEspece(apportEnergie: 0, travailRendu: 10));
Console.WriteLine("... et est-il donc possible ? -> " +
    !Conservation.EstPerpetuumMobile1reEspece(0, 10));

// Contre-temoin : un moteur alimente respecte la conservation, il est possible.
Console.WriteLine("Un moteur alimente de 10 J rendant 8 J est-il un perpetuum mobile ? " +
    Conservation.EstPerpetuumMobile1reEspece(apportEnergie: 10, travailRendu: 8));

// --- Verification des valeurs rejouees -------------------------------------
Verification.Verifie("moteur sans apport : perpetuum mobile de 1re espece",
    Conservation.EstPerpetuumMobile1reEspece(apportEnergie: 0, travailRendu: 10), true);
Verification.Verifie("moteur sans apport : la construction est impossible",
    !Conservation.EstPerpetuumMobile1reEspece(0, 10), false);
Verification.Verifie("moteur alimente : pas de perpetuum mobile",
    Conservation.EstPerpetuumMobile1reEspece(apportEnergie: 10, travailRendu: 8), false);
Verification.Verifie("apport egal au travail : aucune energie creee",
    Conservation.EstPerpetuumMobile1reEspece(apportEnergie: 10, travailRendu: 10), false);


Un moteur qui rend 10 J sans en recevoir est-il un perpetuum mobile ? True


... et est-il donc possible ? -> False


Un moteur alimente de 10 J rendant 8 J est-il un perpetuum mobile ? False


## L'information comme support

L'information est définie **sans observateur**, par le **support d'information** :
« un système dont les états permettent **toutes les permutations** et aussi la **tâche
de copie** ». L'**interopérabilité** est le principe fort : deux systèmes qui sont chacun
un support d'information, composés, forment encore un support, de plus grande capacité.

C'est ce principe qui relie la théorie du constructeur au **no-cloning** : la copie d'un
état quantique inconnu est une tâche **impossible**, et le site la cite comme le
phénomène caractéristique que la théorie de l'information (Deutsch & Marletto, 2015)
ramène à une seule propriété constructeur.

In [3]:
// Illustration 3 -- support d'information et no-cloning
// Support d'information : toutes les permutations possibles ET la copie possible.

static class Information
{
    public static bool EstSupportInformation(bool permutationsPossibles, bool copiePossible)
        => permutationsPossibles && copiePossible;
}

Console.WriteLine("Bit classique (2 etats, on peut tout permuter et copier) : " +
    Information.EstSupportInformation(permutationsPossibles: true, copiePossible: true));
Console.WriteLine("Qubit inconnu (permutations ok, mais copie de l'inconnu IMPOSSIBLE) : " +
    Information.EstSupportInformation(permutationsPossibles: true, copiePossible: false));

// --- Verification des valeurs rejouees -------------------------------------
Verification.Verifie("bit classique : support d'information",
    Information.EstSupportInformation(permutationsPossibles: true, copiePossible: true), true);
Verification.Verifie("qubit inconnu : copie impossible, donc pas de support d'information",
    Information.EstSupportInformation(permutationsPossibles: true, copiePossible: false), false);
Verification.Verifie("permutations impossibles : pas de support d'information",
    Information.EstSupportInformation(permutationsPossibles: false, copiePossible: true), false);


Bit classique (2 etats, on peut tout permuter et copier) : True


Qubit inconnu (permutations ok, mais copie de l'inconnu IMPOSSIBLE) : False


## Le pont vers la série et au-delà

- **→ `Jauge/`** : le théorème de Noether associe à chaque symétrie continue une quantité
  conservée. Lue en langage constructeur, une conservation est un énoncé de **tâche
  impossible** (certains constructeurs sont exclus) — le même type de principe que la
  conservation de l'énergie ci-dessus.
- **→ `IIT/ICT-Series/` (causal bridges)** : le principe de **localité** du site — « une
  transformation qui agit sur A *seul* ne peut changer B » — est le contrepoint
  constructeur de la structure causale ; le site insiste sur la localité (pas
  « causalité ») et rejette le vocabulaire de « non-localité » de type Bell.
- **→ hybridation quantique** : le cas d'école du site est la **gravité** : « si une
  masse est quantique, elle peut être en deux lieux à la fois, et cela brise la
  structure de la relativité générale ». Un médiateur classique ne peut pas
  intriquer deux systèmes quantiques — c'est exactement ce que l'illustration suivante
  met en simulation.

## Le test de la gravité en simulation : le témoin de non-classicité

Le cas d'école du pont ci-dessus — la **gravité** comme médiateur — est aussi le
test le plus abouti de la théorie du constructeur. Le **théorème général des
témoins** (General Witness Theorem) s'énonce ainsi : si un **médiateur** M (par
exemple la gravité) parvient à **intriquer** deux systèmes quantiques Q_A et Q_B
par des **interactions locales** (Q_A ne parle qu'à M, M à Q_B, jamais Q_A à Q_B
directement), alors M **ne peut pas être classique**. Le **test BMV**
(Bose-Marletto-Vedral) en est l'application : deux masses, chacune dans une
superposition de positions, reliées seulement par leur champ gravitationnel ; une
intrication observée entre elles rendrait la gravité non-classique.

Ces tests sont recensés et discutés par Marletto, Deutsch & Vedral dans *Tests of
constructor theory* (2606.07352), qui y passe aussi en revue les variantes
optomécaniques et les objections.

La cellule suivante rejoue la mesure : deux qubits dans une superposition de
positions, un médiateur qui agit sur eux, et la **négativité** de la transposée
partielle comme quantité d'intrication. En C#, les quatre matrices 4×4 sont
écrites à la main et les valeurs propres viennent d'une rotation de Jacobi — BCL
seule, sans `MathNet`, pour que la mesure reste lisible de bout en bout.

In [4]:
// Illustration 4 -- le test BMV de la gravite, en simulation (temoin de non-classicite)
// Deux sondes quantiques (qubits), chacune dans une superposition de deux positions.
// Un mediateur M les relie ; chacune n'interagit qu'avec M (jamais entre elles).
// Theoreme general (2606.07352) : si M intrique les deux sondes par de telles
// interactions locales, M n'est PAS classique.
using System;
using System.Globalization;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;

// Algebre 4x4 et mesure d'intrication -- BCL seule, aucune dependance externe.
static class Quantique
{
    // Superposition des 2 positions : |x+> = (|0> + |1>) / sqrt(2).
    public static double[] EtatX()
    {
        double s = 1.0 / Math.Sqrt(2.0);
        return new[] { s, s };
    }

    // Produit tensoriel de deux vecteurs de dimension 2 -> vecteur de dimension 4, index i*2+j.
    public static double[] Kron(double[] a, double[] b)
    {
        var r = new double[a.Length * b.Length];
        for (int i = 0; i < a.Length; i++)
            for (int j = 0; j < b.Length; j++)
                r[i * b.Length + j] = a[i] * b[j];
        return r;
    }

    // Produit exterieur |v><v| : matrice de densite d'un etat pur.
    public static double[,] Outer(double[] v)
    {
        var r = new double[v.Length, v.Length];
        for (int i = 0; i < v.Length; i++)
            for (int j = 0; j < v.Length; j++)
                r[i, j] = v[i] * v[j];
        return r;
    }

    public static double[,] Diag(params double[] d)
    {
        var r = new double[d.Length, d.Length];
        for (int i = 0; i < d.Length; i++) r[i, i] = d[i];
        return r;
    }

    public static double[,] Produit(double[,] a, double[,] b)
    {
        int n = a.GetLength(0);
        var r = new double[n, n];
        for (int i = 0; i < n; i++)
            for (int j = 0; j < n; j++)
            {
                double s = 0.0;
                for (int k = 0; k < n; k++) s += a[i, k] * b[k, j];
                r[i, j] = s;
            }
        return r;
    }

    // Melange pondere de deux matrices de densite : (1-w1)*x0 + w1*x1.
    public static double[,] Melange(double[,] x0, double[,] x1, double w1)
    {
        var r = new double[4, 4];
        for (int i = 0; i < 4; i++)
            for (int j = 0; j < 4; j++)
                r[i, j] = (1 - w1) * x0[i, j] + w1 * x1[i, j];
        return r;
    }

    // Transposee partielle sur B : rho^T_B[(a,b),(c,d)] = rho[(a,d),(c,b)].
    public static double[,] TransposeePartielleB(double[,] rho)
    {
        var rt = new double[4, 4];
        for (int a = 0; a < 2; a++)
            for (int b = 0; b < 2; b++)
                for (int c = 0; c < 2; c++)
                    for (int d = 0; d < 2; d++)
                        rt[a * 2 + b, c * 2 + d] = rho[a * 2 + d, c * 2 + b];
        return rt;
    }

    // Valeurs propres d'une matrice symetrique reelle par rotations de Jacobi.
    public static double[] ValeursPropresSymetriques(double[,] m)
    {
        int n = m.GetLength(0);
        var a = (double[,])m.Clone();
        for (int balayage = 0; balayage < 100; balayage++)
        {
            double max = 0.0; int p = 0, q = 1;
            for (int i = 0; i < n; i++)
                for (int j = i + 1; j < n; j++)
                    if (Math.Abs(a[i, j]) > max) { max = Math.Abs(a[i, j]); p = i; q = j; }
            if (max < 1e-14) break;

            double theta = (a[q, q] - a[p, p]) / (2.0 * a[p, q]);
            double t = theta >= 0
                ? 1.0 / (theta + Math.Sqrt(theta * theta + 1.0))
                : -1.0 / (-theta + Math.Sqrt(theta * theta + 1.0));
            double cos = 1.0 / Math.Sqrt(t * t + 1.0);
            double sin = t * cos;

            for (int k = 0; k < n; k++)
            {
                double akp = a[k, p], akq = a[k, q];
                a[k, p] = cos * akp - sin * akq;
                a[k, q] = sin * akp + cos * akq;
            }
            for (int k = 0; k < n; k++)
            {
                double apk = a[p, k], aqk = a[q, k];
                a[p, k] = cos * apk - sin * aqk;
                a[q, k] = sin * apk + cos * aqk;
            }
        }
        var propres = new double[n];
        for (int i = 0; i < n; i++) propres[i] = a[i, i];
        Array.Sort(propres);
        return propres;
    }

    // Mesure d'intrication : (||rho^T_B||_1 - 1)/2, nulle pour un etat separable.
    public static double Negativite(double[,] rho)
    {
        var propres = ValeursPropresSymetriques(TransposeePartielleB(rho));
        double somme = 0.0;
        foreach (var l in propres) somme += Math.Abs(l);
        return Math.Max(0.0, 0.5 * (somme - 1.0));
    }

    public const double SeuilIntrication = 0.0;   // negativite strictement positive => intrication

    // Etiquette derivee de la negativite CALCULEE, jamais ecrite a l'avance.
    public static string VerdictNonClassicite(double n)
        => n > SeuilIntrication ? "NON-CLASSIQUE" : "classique";
}

var rhoProbes = Quantique.Outer(Quantique.Kron(Quantique.EtatX(), Quantique.EtatX()));
var cz = Quantique.Diag(1.0, 1.0, 1.0, -1.0);                  // mediateur QUANTIQUE : interaction de type Ising
var zz = Quantique.Diag(1.0, -1.0, -1.0, 1.0);                 // phase locale commune (classique)

// 1) Mediateur quantique : l'interaction intrique les deux sondes.
var rhoQuantique = Quantique.Produit(Quantique.Produit(cz, rhoProbes), cz);
double nQuantique = Quantique.Negativite(rhoQuantique);

// 2) Mediateur classique (bit) : transmet une phase locale commune, moyennee sur
//    l'inconnue classique (50/50) -> correlation classique, PAS d'intrication.
var rhoClassique = Quantique.Melange(rhoProbes, Quantique.Produit(Quantique.Produit(zz, rhoProbes), zz), 0.5);
double nClassique = Quantique.Negativite(rhoClassique);

Console.WriteLine("Temoin de non-classicite (2606.07352) : le mediateur intrique-t-il les sondes ?");
Console.WriteLine($"  mediateur quantique : negativite = {nQuantique:F3}  -> {Quantique.VerdictNonClassicite(nQuantique)}");
Console.WriteLine($"  mediateur classique : negativite = {nClassique:F3}  -> {Quantique.VerdictNonClassicite(nClassique)}");
Console.WriteLine();
Console.WriteLine("Transition qubit -> bit (le mediateur perd sa coherence quantique) :");
foreach (var p in new[] { 0.0, 0.25, 0.5, 0.75, 1.0 })
{
    var rho = Quantique.Melange(rhoQuantique, rhoClassique, p);
    Console.WriteLine($"  part classique p={p:F2} : negativite = {Quantique.Negativite(rho):F3}");
}

// --- Verification des valeurs rejouees -------------------------------------
// Exactitude : la negativite du mediateur quantique vaut exactement 1/2 (etat
// maximally intrique : la transposee partielle a pour valeurs propres
// -1/2, 1/2, 1/2, 1/2) et celle du mediateur classique exactement 0 (spectre
// 0, 0, 1/2, 1/2). Les cibles sont portees par des rationnels exacts reduits ;
// la comparaison avec la valeur calculee passe par une tolerance explicite,
// jamais par une egalite flottante.
const double ToleranceExacte = 1e-12;

var propresQuantique = Quantique.ValeursPropresSymetriques(Quantique.TransposeePartielleB(rhoQuantique));
var propresClassique = Quantique.ValeursPropresSymetriques(Quantique.TransposeePartielleB(rhoClassique));

Verification.Verifie("negativite du mediateur quantique (exacte 1/2)", nQuantique, new Rational(1, 2), ToleranceExacte);
Verification.Verifie("negativite du mediateur classique (exacte 0)", nClassique, new Rational(0, 1), ToleranceExacte);
Verification.Verifie("spectre transposee partielle, mediateur quantique [0] (exact -1/2)",
    propresQuantique[0], new Rational(-1, 2), ToleranceExacte);
Verification.Verifie("spectre transposee partielle, mediateur quantique [1] (exact 1/2)",
    propresQuantique[1], new Rational(1, 2), ToleranceExacte);
Verification.Verifie("spectre transposee partielle, mediateur quantique [2] (exact 1/2)",
    propresQuantique[2], new Rational(1, 2), ToleranceExacte);
Verification.Verifie("spectre transposee partielle, mediateur quantique [3] (exact 1/2)",
    propresQuantique[3], new Rational(1, 2), ToleranceExacte);
Verification.Verifie("spectre transposee partielle, mediateur classique [0] (exact 0)",
    propresClassique[0], new Rational(0, 1), ToleranceExacte);
Verification.Verifie("spectre transposee partielle, mediateur classique [1] (exact 0)",
    propresClassique[1], new Rational(0, 1), ToleranceExacte);
Verification.Verifie("spectre transposee partielle, mediateur classique [2] (exact 1/2)",
    propresClassique[2], new Rational(1, 2), ToleranceExacte);
Verification.Verifie("spectre transposee partielle, mediateur classique [3] (exact 1/2)",
    propresClassique[3], new Rational(1, 2), ToleranceExacte);
Verification.Verifie("verdict du mediateur quantique", Quantique.VerdictNonClassicite(nQuantique), "NON-CLASSIQUE");
Verification.Verifie("verdict du mediateur classique", Quantique.VerdictNonClassicite(nClassique), "classique");

// La transition qubit -> bit : la negativite decroit strictement quand la part
// classique augmente, et s'annule a p = 1. Les valeurs intermediaires sont
// irrationnelles (racines d'un polynome) : la tolerance vaut la demi-unite de
// la derniere decimale affichee par la source (5e-4).
const double ToleranceAffichee = 5e-4;
double[] parties = { 0.0, 0.25, 0.5, 0.75, 1.0 };
double[] negativitesAttendues = { 0.500, 0.318, 0.155, 0.038, 0.000 };
double precedente = double.NaN;
for (int i = 0; i < parties.Length; i++)
{
    double n = Quantique.Negativite(Quantique.Melange(rhoQuantique, rhoClassique, parties[i]));
    Verification.Verifie($"negativite a p = {parties[i]:F2}", n, negativitesAttendues[i], ToleranceAffichee);
    if (i == 0)
        Verification.Verifie("negativite a p = 0.00 (exacte 1/2)", n, new Rational(1, 2), ToleranceExacte);
    if (i == parties.Length - 1)
        Verification.Verifie("negativite a p = 1.00 (exacte 0)", n, new Rational(0, 1), ToleranceExacte);
    if (i > 0 && !(n < precedente))
        throw new InvalidOperationException(
            $"La negativite ne decroit pas strictement entre p = {parties[i - 1]:F2} et p = {parties[i]:F2}.");
    precedente = n;
}


Temoin de non-classicite (2606.07352) : le mediateur intrique-t-il les sondes ?


  mediateur quantique : negativite = 0.500  -> NON-CLASSIQUE


  mediateur classique : negativite = 0.000  -> classique


Transition qubit -> bit (le mediateur perd sa coherence quantique) :


  part classique p=0.00 : negativite = 0.500


  part classique p=0.25 : negativite = 0.318


  part classique p=0.50 : negativite = 0.155


  part classique p=0.75 : negativite = 0.038


  part classique p=1.00 : negativite = 0.000


## Exercices

In [5]:
// Exercice 1 -- accomplir une tache
// Retourne l'etat de sortie si le constructeur amene tache.Entree vers
// tache.Sortie, sinon null (tache non accomplie par ce constructeur).
Func<Tache, Constructeur, string> accomplir = null;   // TODO etudiant

Console.WriteLine("Exercice a completer");

Exercice a completer


In [6]:
// Exercice 2 -- est-ce un constructeur ?
// Un constructeur "reste le meme" apres la tache : sa capacite a transformer est
// conservee. Retourne true si l'objet est inchange, false sinon, null si non defini.
Func<string, string, bool> estConstructeur = null;   // TODO etudiant

Console.WriteLine("Exercice a completer");

Exercice a completer


In [7]:
// Exercice 3 -- l'interoperabilite
// Deux supports d'information composes forment-ils encore un support ?
// (indice : c'est le principe d'interoperabilite de l'Illustration 3)
// supportA et supportB sont des etiquettes de supports ("bit", "qubit inconnu", ...).
Func<string, string, bool> interoperable = null;   // TODO etudiant

Console.WriteLine("Exercice a completer");

Exercice a completer


## Ce que ce carnet ne couvre pas

Restitué fidèlement au site : la **mécanique** du no-cloning (son énoncé et sa preuve
requièrent la théorie quantique — cf. le papier de l'information), la **deuxième loi**
de la thermodynamique (la « théorie du constructeur de la thermodynamique » est annoncée
mais non développée sur le site), et la **causalité** en tant que telle (le site ne
traite que la localité).

## À retenir

| Notion | Ce qu'elle dit | Où la voir |
|---|---|---|
| Tâche et constructeur | une tâche est une transformation d'attribut (3 → 4 sur le dé) ; un constructeur l'accomplit **et reste le même**, donc il peut la répéter : `c_3vers4` l'accomplit (`true`), le constructeur « rien » non (`false`) | Illustration 1 |
| Possible / impossible | une tâche est impossible si **aucun** constructeur ne peut l'accomplir à une précision arbitraire ; les lois de la théorie portent sur les tâches, pas sur les constructeurs | Le vocabulaire |
| Inversion | les lois deviennent des principes sur ce qui peut, ou non, être produit ; la description dynamique (conditions initiales et lois de mouvement) se retrouve dans de nombreux cas, elle n'est pas remplacée | L'inversion |
| Conservation | la conservation de l'énergie exclut un constructeur : le moteur qui rend 10 J sans en recevoir est un perpétuum mobile de première espèce (`true`), donc impossible (`false`) | Illustration 2 |
| Support d'information | permutations **et** copie possibles : le bit classique en est un (`true`), le qubit inconnu non (`false`), parce que copier un état quantique inconnu est une tâche impossible (no-cloning) | Illustration 3 |
| Témoin de non-classicité | un médiateur qui intrique deux sondes par des interactions seulement locales n'est pas classique : négativité 0,500 avec le médiateur quantique, 0,000 avec le médiateur classique, et 0,318 · 0,155 · 0,038 quand la part classique $p$ passe de 0,25 à 0,75 | Illustration 4 |

Les quatre illustrations sont des **systèmes jouets** : elles mettent le vocabulaire en action, elles ne démontrent pas les principes. Les booléens des illustrations 1 à 3 suivent directement des définitions codées ; seule l'illustration 4 calcule une quantité (la négativité, par transposée partielle) dont la valeur n'est pas écrite d'avance dans le code — et l'étiquette de non-classicité qui l'accompagne en est **déduite du seuil** (négativité strictement positive), non écrite d'avance non plus.

Le portage C# suit le même partage : les illustrations 1 à 3 sont des définitions exécutables, l'illustration 4 porte le seul calcul numérique du carnet (algèbre 4×4 et valeurs propres par rotations de Jacobi, BCL seule, sans dépendance externe).

## Vérifiez votre compréhension

1. Le constructeur « rien » (règle vide) n'accomplit pas la tâche 3 → 4. Pourquoi cela ne suffit-il **pas** à dire que la tâche est impossible ?

<details><summary>Réponse</summary>

Parce qu'« impossible » signifie qu'**aucun** constructeur ne peut accomplir la tâche à une précision arbitraire. Un échec particulier ne dit rien de tous les constructeurs : `c_3vers4` l'accomplit (`true`), la tâche est donc possible. L'impossibilité est un énoncé sur la classe entière des constructeurs : c'est pour cela que la théorie l'énonce comme une loi (conservation, no-cloning), et non comme le résultat d'un essai.

</details>

2. En quoi la conservation de l'énergie, dans ce langage, ressemble-t-elle au no-cloning ?

<details><summary>Réponse</summary>

Les deux sont des énoncés de **tâche impossible**. L'une exclut le constructeur qui rend du travail sans apport d'énergie (illustration 2 : possible → `false`) ; l'autre exclut la copie d'un état quantique inconnu (illustration 3 : le qubit inconnu n'est pas un support d'information → `false`). Aucun des deux ne décrit une trajectoire : tous deux disent ce qui ne peut pas être produit, quelle que soit la machine envisagée. C'est aussi la lecture que le pont vers `Jauge/` propose pour les conservations de Noether.

</details>

3. Dans la simulation BMV, que montre la décroissance de la négativité de 0,500 à 0,000 quand la part classique $p$ passe de 0 à 1, et que ne montre-t-elle **pas** ?

<details><summary>Réponse</summary>

Elle montre que l'intrication des sondes s'efface à mesure que le médiateur perd sa cohérence quantique : un médiateur entièrement classique ($p = 1$) n'en crée aucune, comme l'exige le théorème général des témoins. La transition est progressive (0,318 · 0,155 · 0,038) : tant que la négativité reste strictement positive, les sondes sont intriquées, et le théorème interdit qu'un médiateur entièrement classique l'ait produit. Ce qu'elle ne montre pas : que la gravité est quantique. Le médiateur est ici un modèle choisi (une interaction de type Ising, `CZ`), pas la gravité ; seule une expérience BMV réelle pourrait trancher.

</details>

## Pour aller plus loin

- **Sous-série `Jauge/`** : [Jauge-01 — Théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb) — la conservation y est **mesurée** (courant de Noether, dérive numérique, contre-témoin anisotrope) ; la relire ensuite en langage constructeur, comme un énoncé de tâche impossible.
- **Localité et structure causale** : [IIT / ICT-Series](../../IIT/ICT-Series/README.md) — les trajectoires de structures causales, contrepoint du principe de localité du site (« une transformation qui agit sur A *seul* ne peut changer B »).
- **Jumeau Python** : [Constructeur-01 — les lois comme tâches (Python)](Constructeur-01-Lois-Comme-Taches-Python.ipynb) — même plan, mêmes valeurs ; les deux carnets doivent rendre la même négativité.
- **Plan de la sous-série** : [README Constructeur](README.md).
- **Fil de la série** : [#23161 — EPIC Série Physique numérique](https://github.com/jsboige/CoursIA/issues/23161).

## Références

- D. Deutsch, *The Philosophy of Constructor Theory*, **Synthese** 190(18), 2013.
  [doi:10.1007/s11229-013-0279-z](https://doi.org/10.1007/s11229-013-0279-z) ·
  [arXiv:1210.7439](https://arxiv.org/abs/1210.7439)
- D. Deutsch, C. Marletto, *The Constructor Theory of Information*,
  **Proc. R. Soc. A** 471(2174), 2015.
  [doi:10.1098/rspa.2014.0540](https://doi.org/10.1098/rspa.2014.0540) ·
  [arXiv:1405.5563](https://arxiv.org/abs/1405.5563)
- C. Marletto, D. Deutsch, V. Vedral, *Tests of constructor theory*, 2026.
  [arXiv:2606.07352](https://arxiv.org/abs/2606.07352)
- Site du projet : <https://www.constructortheory.org/> (pages *What is*, *FAQ*,
  *Applications* — lues et ancrées pour ce carnet).